# SplatDiff: CUDA vs OpenMP vs serial benchmark (Colab)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. Takes about 5 minutes.
At the end, download `results/benchmark.json` (last cell) and commit it to the repo, or paste it to Claude.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!nproc && lscpu | grep 'Model name'

In [ ]:
!git clone -q https://github.com/AakarshP26/SplatDIff.git
%cd SplatDIff
!git log --oneline | head -3

In [ ]:
# scenes: download, convert (Node is preinstalled on Colab), generate test cases
!bash scripts/convert.sh guitar biker
!python3 python/prepare.py

In [ ]:
# build serial, OpenMP and CUDA
!make -C kernels
!nvcc -O3 -arch=sm_75 --use_fast_math -o kernels/cuda/splatdiff_cuda kernels/cuda/splatdiff.cu

In [ ]:
# correctness: every implementation vs the NumPy reference
!python3 python/test_kernels.py guitar move 48
!python3 python/test_kernels.py biker reparam 48

In [ ]:
# speedup study (writes results/benchmark.json)
!python3 python/benchmark.py --sizes 64 128 256 --scene biker --case move --reps 3

In [ ]:
import json
b = json.load(open('results/benchmark.json'))
for r in b['runs']:
    print(f"N={r['n']:4d} {r['impl']:7s} threads={r.get('threads','-')!s:3s} total {r['total']*1e3:9.2f} ms  speedup {r['speedup']:6.1f}x")

In [ ]:
from google.colab import files
files.download('results/benchmark.json')